In [0]:
class BasePreprocessor(ABC):
    @abstractmethod
    def process(self):
        """
        Returns
        -------
        pyspark.sql.DataFrame
            Processed dataframe.
        """
        pass

In [0]:
class SalesProcess(BasePreprocessor):
    def __init__(self,raw_data,filter_map):
        self.raw_data = raw_data
        self.filter_map = filter_map
        self.raw_sales = self.raw_data['raw_sales']
        self.calendar_data = self.raw_data['calendar']
    
    def unpivot_sales_data(self):
        self.sales_long = self.raw_sales\
                            .unpivot(ids=self.filter_map['sales_id_cols'],
                                    values = [c for c in self.raw_sales.columns if c.startswith('d_')],
                                            variableColumnName = "d",
                                            valueColumnName = "sales")
                            
        self.sales_long = self.sales_long.withColumn('id',F.regexp_replace(F.col('id'),'_evaluation',""))
        return self.sales_long
    
    def join_sales_calndar(self):
        self.calendar_filtered = self.calendar_data.select('d','date').distinct()
        self.sales_processed = self.sales_long.join(self.calendar_filtered,on=['d'],how='left')
        return self.sales_processed
    
    def process(self):
        self.sales_long = self.unpivot_sales_data()
        self.sales_processed = self.join_sales_calndar()
        return self.sales_processed

In [0]:
class EventsProcess(BasePreprocessor):
    def __init__(self,raw_data,filter_map):
        self.raw_data = raw_data
        self.filter_map = filter_map
        self.events_data = self.raw_data['calendar'].select('d','date','year','event_name_1','event_name_2','snap_CA','snap_WI','snap_TX').distinct()
        self.event_flag_cond = ((F.col('event_name_1').isNotNull())|(F.col('event_name_2').isNotNull()))

    def normalize_event_name(self, event):
        return (
            event.lower()
            .replace(" ", "_")
            .replace("'", "")
            .replace("-", "_")
        )

    def encode_events(self,events_data):  
        events_data = events_data.withColumn('event_flag',
                                             F.when(self.event_flag_cond,F.lit(1)).otherwise(F.lit(0)))\
                            .withColumn('event_flag',F.col('event_flag').cast('byte'))
        event1_values = [r["event_name_1"] for r in events_data.select("event_name_1").distinct().collect() if r["event_name_1"] is not None]
        event2_values = [r["event_name_2"] for r in events_data.select("event_name_2").distinct().collect() if r["event_name_2"] is not None]

        all_events = set(event1_values) | set(event2_values)

        event_exprs = {}
        for event in all_events:
            normalized_event = self.normalize_event_name(event)
            event_exprs[f"f_event_{normalized_event}"] = F.when(
                (F.col("event_name_1") == event) |
                (F.col("event_name_2") == event),
                1
            ).otherwise(0)

        events_encoded = events_data.withColumns(event_exprs)
        
        return events_encoded
    
    def process(self):
        self.events_processed = self.encode_events(self.events_data)
        return self.events_processed


    


In [0]:
class PriceProcess(BasePreprocessor):
    def __init__(self,raw_data,filter_map):
        self.raw_data = raw_data
        self.filter_map = filter_map
        self.price_data = self.raw_data['raw_price']
        self.calendar = self.raw_data['calendar'].select('d','date','wm_yr_wk').distinct()

    def process(self):
        processed_price = self.price_data.join(self.calendar,on=['wm_yr_wk'],how='left').select('store_id','item_id','sell_price','d','date')
        return processed_price

In [0]:
class JoinedDataProcess(BasePreprocessor):
    def __init__(self,filter_map,sales,events,price):
        self.filter_map = filter_map
        self.sales = sales
        self.events = events
        self.price = price
    
    def transform_data(self):
        joined_data = self.sales.join(self.events,on=['d','date'],how='left')
        joined_data = joined_data.join(self.price,on=['d','date','store_id','item_id'],how='left')\
                                            .withColumn('week',F.weekofyear(F.col('date')))\
                                            .withColumn('week',F.lpad(F.col('week'),2,'0'))\
                                            .withColumn('yearweek',F.concat(F.col('year'),F.col('week')))

        return joined_data
    
    def agg_data(self,joined_data):

        group_cols = ['id','item_id','dept_id','cat_id','store_id','state_id','yearweek']
        
        event_cols = [c for c in joined_data.columns if c.startswith('f_event_')]
        
        event_aggs = [F.max(F.col(c).cast('byte')).alias(c) for c in event_cols]

        weekly_data = joined_data.groupby(*group_cols).agg(
            F.sum('sales').cast('float').alias('sales'),
            # Price is static per item per wm_yr_wk in M5, so first() is used
            F.first('sell_price').cast('float').alias('sell_price'), 
            # Sum the SNAP flags to get the count of SNAP days in that week (0 to 7)
            F.sum(F.col('snap_CA')).cast('byte').alias('snap_CA_days'),
            F.sum(F.col('snap_TX')).cast('byte').alias('snap_TX_days'),
            F.sum(F.col('snap_WI')).cast('byte').alias('snap_WI_days'),
            F.max('event_flag').cast('byte').alias('event_flag'),
            *event_aggs
        )
        return weekly_data
    
    def leading_zero_sales_removal(self,joined_data):
        window_spec = (Window.partitionBy(self.filter_map['key_col']).orderBy(self.filter_map['yearweek_col']).rowsBetween(Window.unboundedPreceding,Window.currentRow))
        joined_data = joined_data\
                .withColumn('cumsum_sales',F.sum(F.coalesce(F.col(self.filter_map['sales_col']), F.lit(0))).over(window_spec))\
                .filter(F.col("cumsum_sales") > 0)\
                .drop("cumsum_sales")
        return joined_data
    
    def process(self):
        transformed_data = self.transform_data()
        aggregated_data = self.agg_data(transformed_data)
        joined_data = self.leading_zero_sales_removal(aggregated_data).filter(F.col('yearweek')>="201301")
        write_data(joined_data,processed_paths['joined_data'],["state_id"])
        return joined_data




In [0]:
class FeatureEngineering:
    def __init__(self,joined_data,filter_map,m5_config):
        self.joined_data = joined_data
        self.filter_map = filter_map
        self.m5_config = m5_config
    
    def creat_lag_features(self):
        """computes the sales lags defined in m5_config."""
        id = self.m5_config['id_col']
        lags = self.m5_config['lags_features']
        week = self.m5_config['week_col']
        y = self.m5_config['y']

        df = (self.joined_data[[id,week,y]].drop_duplicates().sort_values(by=[id,week]))

        for lag in lags:
            df[f"{y}_lag_{lag}"] = df.groupby(id)[y].shift(lag)
        
        return self.joined_data.merge(df.drop(columns=[y]),on=[id,week],how='left')
        
